<a href="https://colab.research.google.com/github/timfitz04/Business-Analytics-Dissertation/blob/main/MeathWS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://meath.gaa.ie/fixtures-results/

In this file Meath league tables were scraped and collected then stored.

Team names were then cleaned so the dataset would  merge with gaapitchfinder dataset adding longitudal and latiudal data to the orignial data that was scraped. Teams that did not match after cleaning were mannually matched.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
os.chdir("/content/drive/MyDrive/Dissertation/Meath")
!pwd

/content/drive/MyDrive/Dissertation/Meath


In [ ]:
!pip install bs4

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import time
time.sleep(5)

compIDs_by_year = {
    2024: {
        "Division 1": 211465,
        "Division 2": 211467,
        "Division 3": 211469,
        "Division 4": 211471,
        "Division 5": 211473,
        "Division 6": 211495,
        "Division 7": 211497,
        "Division 8": 211499,
        "Division 9": 211501,
        "Division 10": 211503,
    },
    2023: {
        "Division 1 A": 189303,
        "Division 1 B": 189305,
        "Division 2 A": 189307,
        "Division 2 B": 189309,
        "Division 3 A": 189311,
        "Division 3 B": 189313
    },
    2022: {
        "Division 1 A": 170713,
        "Division 1 B": 170715,
        "Division 2 A": 170717,
        "Division 2 B": 170719,
        "Division 3 A": 170721,
        "Division 3 B": 170723,
        "Division 4": 170725},
    2021: {
        "Division 1 A": 156787,
        "Division 1 B": 156789,
        "Division 2 A": 156793,
        "Division 2 B": 156795,
        "Division 2 C": 156797,},

    2019: {
        "A-League Division 1": 113731,
        "A-League Division 2": 113733,
        "A-League Division 3": 113735,
        "A-League Division 4": 113737,
        "B-League Division 1": 116121,
        "B-League Division 2": 116123,
        "B-League Division 3": 116125,
        "B-League Division 4": 116127,
        "B-League Division 5": 116129,
        "B-League Division 6": 116131},
    2018: {
        "A-League Division 1": 95002,
        "A-League Division 2": 95004,
        "A-League Division 3": 95006,
        "A-League Division 4": 95008,
        "B-League Division 1": 96907,
        "B-League Division 2": 96909,
        "B-League Division 3": 96911,
        "B-League Division 4": 96913,
        "B-League Division 5": 96915,
        "B-League Division 6": 96917}
}

all_rows = []

for year, divisions in compIDs_by_year.items():

    for division_name, compid in divisions.items():

        url = (
                "https://meath.gaa.ie/fixtures-results/fixtures-results-ajax/"
                f"?countyBoardID=22&ccAC=1&compID={compid}"
                "&leagueTable=Y&resultsOnly=Y&reverseDateOrder=Y&orderTBCLast=Y"
        )

        response = requests.get(url)
        time.sleep(1)
        soup = BeautifulSoup(response.text, "html.parser")


        tables = pd.read_html(str(soup))
        if len(tables) == 0:
            print(f"No table for {division_name} {year}")

            continue

        df = tables[0]



        df.columns = (
            df.columns
            .str.strip()
            .str.lower()
            .str.replace(" ", "_")
        )

        rename_map = {
            "p": "pld",
            "team": "team",
            "w": "w",
            "d": "d",
            "l": "l",
            "f": "pf",
            "a": "pa",
            "pts": "pts"
        }
        df = df.rename(columns=rename_map)


        # Calculate points difference
        df["pd"] = df["pf"].astype(int) - df["pa"].astype(int)

        # Add division + year
        df["Division"] = division_name
        df["Year"] = year
        df["County"] = "Meath"

        df = df[["County", "Division", "Year", "team", "pld", "w", "d", "l", "pf", "pa", "pd", "pts"]]

        all_rows.append(df)


# ---- FINAL OUTPUT ----
Meath = pd.concat(all_rows, ignore_index=True)

print(Meath)


/tmp/ipython-input-4248479189.py:86: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(str(soup))
/tmp/ipython-input-4248479189.py:86: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(str(soup))
/tmp/ipython-input-4248479189.py:86: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(str(soup))
/tmp/ipython-input-4248479189.py:86: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(str(soup))
/tmp/ipython-input-4248479189.py:86:

    County             Division  Year                  team  pld  w  d  l  \
0    Meath           Division 1  2024                Skryne   11  7  1  3   
1    Meath           Division 1  2024            Summerhill   11  7  1  3   
2    Meath           Division 1  2024        Ballinabrackey   11  5  4  2   
3    Meath           Division 1  2024  Donaghmore/Ashbourne   11  6  2  3   
4    Meath           Division 1  2024               Ratoath   11  5  3  3   
..     ...                  ...   ...                   ...  ... .. .. ..   
498  Meath  B-League Division 6  2018            Kilmainham    7  4  0  3   
499  Meath  B-League Division 6  2018           Drumconrath    7  3  0  4   
500  Meath  B-League Division 6  2018        Kilmainhamwood    7  2  0  5   
501  Meath  B-League Division 6  2018               Moylagh    7  2  0  5   
502  Meath  B-League Division 6  2018                 Slane    6  2  0  4   

      pf   pa  pd  pts  
0    151  133  18   15  
1    168  155  13   15  


/tmp/ipython-input-4248479189.py:86: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(str(soup))


In [ ]:
import re
def clean_club_name(name):
    name = name.lower()

    # remove trailing numbers (Thomas Davis 2 → Thomas Davis)
    name = re.sub(r"\s*\d+$", "", name)

    # remove reserve/second team indicators e.g. "2nd", "B", "C"
    name = re.sub(r"\b(2nd|second|team|b|c)\b", "", name, flags=re.IGNORECASE)

    # strip GAA-specific suffixes and organisation labels
    name = re.sub(r"\b(gaa|clg|club|lgfa|ladies)\b", "", name, flags=re.IGNORECASE)

    # remove punctuation + multiple spaces
    name = re.sub(r"[^\w\s]", "", name)
    name = " ".join(name.split())

    return name.strip()

# after scraping into Dub dataframe:
Meath["clean_team"] = Meath["team"].apply(clean_club_name)

In [ ]:
coords = pd.read_csv("gaapitchfinder_data.csv")
coords["clean_team"] = coords["Club"].apply(clean_club_name)
coords = coords[coords["County"] == "Meath"].copy()
coords = coords.drop_duplicates(
    subset="clean_team",
    keep="first"
)
coords = coords[["clean_team", "Latitude", "Longitude"]]


In [ ]:

manual_map = {
    "bhulf tón": "wolfe tones",
    "simonstown": "simonstown gaels",
    "navan omahonys": "omahonys navan",
    "seneschalstown gfc": "seneschalstown",
    "kilbride gfc": "kilbride",
    "ballivor gfc": "ballivor",
    "drumbaragh": "drumbaragh emmetts",
    "moynalty gfc": "moynalty",
    "naomh muire": "st marys donore",
    "st marys": "st marys donore",
    "st marys gfc": "st marys donore",
    "na fianna": "na fianna enfield",
    "st patricks": "st patricks stamullen",
    "st vincents": "st vincents ardcath",
    "st brigids": "st brigids ballinacree",
    "eastern gaels": "eastern gaels gac"
}

Meath["clean_team"] = Meath["clean_team"].replace(manual_map)



In [ ]:
Meath = Meath.merge(coords, on=["clean_team"], how="left")

In [ ]:
Meath[Meath["Latitude"].isna()][["team","clean_team"]].drop_duplicates()

,team,clean_team


In [ ]:
Meath.to_csv("Meath_league_tables.csv", index=False)